# LLM Zero-Shot Sentiment — Multi-Domain

> Part of: *BERT vs LLM vs SenticNet: A Multi-Domain Sentiment Comparison*

This notebook runs GPT-4o-mini zero-shot across the same three domains.

**Before running this notebook:**
- Make sure `.env` exists with `OPENAI_API_KEY` set
- Actual cost from my run: **$0.1334 total** across all three domains

---

No fine-tuning, no examples — pure zero-shot prompting.
The prompt is domain-aware (slightly different framing per domain).

## Setup

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, '../src')
from data_utils import load_all_domains, SEED, DOMAINS
from llm_utils import run_llm_inference, summarize_llm_results, estimate_cost, LLM_MODEL

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path='../.env')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise EnvironmentError('OPENAI_API_KEY not found. Copy .env.example to .env and fill it in.')

client = OpenAI(api_key=OPENAI_API_KEY)
Path('../results').mkdir(exist_ok=True)
Path('../plots').mkdir(exist_ok=True)

print(f'Model: {LLM_MODEL}')
print('Setup complete.')

## Load Datasets

In [ ]:
datasets = load_all_domains(n_per_domain=2000, dataset_dir='../datasets')
for domain, df in datasets.items():
    print(f'{domain}: {len(df)} samples | avg words: {df["word_count"].mean():.0f}')

## Cost Estimate Before Running

Estimating expected API cost based on token counts before committing.

In [ ]:
est_input_per_sample = 150
n_domains = 3
n_samples = 2000

total_est_tokens = est_input_per_sample * n_samples * n_domains
est = estimate_cost(total_est_tokens, n_samples * n_domains)

print('=== PRE-RUN COST ESTIMATE ===')
print(f'Approx input tokens:  {total_est_tokens:,}')
print(f'Estimated total cost: ${est["total_cost"]:.3f}')
print()
print('Actual cost from my completed run: $0.1334 total')
print('Twitter was cheapest ($0.021/domain) due to shorter texts.')
print('IMDb was most expensive ($0.071/domain) due to longer reviews.')

## Run Inference — All Domains

⚠️ **This cell makes real API calls.** Results are cached to `results/llm_{domain}.csv`.

Temperature=0, max_tokens=5, input truncated to 200 words.

In [ ]:
llm_results = {}
summaries = []

for domain, df in datasets.items():
    cache_path = f'../results/llm_{domain}.csv'

    if Path(cache_path).exists():
        print(f'Loading cached {domain} results from {cache_path}')
        llm_df = pd.read_csv(cache_path)
        llm_results[domain] = llm_df
        summary = summarize_llm_results(llm_df, df['label'], domain=domain)
        summaries.append(summary)
        print()
        continue

    print(f'--- {domain.upper()} --- (making API calls)')
    llm_df = run_llm_inference(
        df['text_clean'].tolist(),
        client,
        domain=domain,
        sleep_between=0.05
    )
    llm_df['correct'] = (llm_df['llm_pred'] == df['label'].values)
    llm_df['ground_truth'] = df['label'].values
    llm_df['text'] = df['text_clean'].values
    llm_df['word_count'] = df['word_count'].values
    llm_df['domain'] = domain

    llm_df.to_csv(cache_path, index=False)
    llm_results[domain] = llm_df

    summary = summarize_llm_results(llm_df, df['label'], domain=domain)
    summaries.append(summary)
    print()

## Summary Table + Actual Cost

In [ ]:
summary_df = pd.DataFrame(summaries)

total_input  = sum(llm_results[d]['llm_input_tokens'].sum()  for d in DOMAINS if d in llm_results)
total_output = sum(llm_results[d]['llm_output_tokens'].sum() for d in DOMAINS if d in llm_results)
actual_cost  = estimate_cost(total_input, total_output)

print(f'Total tokens used: {total_input:,} input / {total_output:,} output')
print(f'Actual total cost: ${actual_cost["total_cost"]:.4f}')
print()

summary_df['accuracy'] = summary_df['accuracy'].map('{:.1%}'.format)
summary_df['avg_latency_ms'] = summary_df['avg_latency_ms'].map('{:.0f} ms'.format)
display(summary_df[['domain', 'accuracy', 'avg_latency_ms', 'total_cost', 'n_samples']])

## Latency Comparison by Domain

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for domain in DOMAINS:
    if domain not in llm_results:
        continue
    latencies = llm_results[domain]['llm_latency_s'] * 1000
    ax.hist(latencies, bins=30, alpha=0.5, label=domain)

ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('LLM Latency Distribution by Domain')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/llm_latency_by_domain.png', dpi=120, bbox_inches='tight')
plt.show()

### On Latency — Actual Measurements

My empirical latency measurements confirmed a domain-dependent pattern consistent with input length:

| Domain | Avg Latency | Avg Words | Cost/1k |
|--------|-------------|-----------|----------|
| **IMDb** | 1,109 ms | 233 | $0.035 |
| **Twitter** | 795 ms | 20 | $0.010 |
| **Amazon** | 1,066 ms | 79 | $0.021 |

Twitter was the fastest and cheapest domain, consistent with its short average text length (20 words). IMDb was the slowest and most expensive, reflecting the 200-word truncation being triggered more frequently on longer reviews. Even truncated, the longer input context meaningfully affects both latency and token cost.

I observed significant latency variance within each domain, reflecting network jitter and server-side load variability — this is the fundamental unpredictability cost of API-based inference compared to local deployment.

## Failure Case Analysis

In [ ]:
def show_llm_failures(llm_df, domain, n=5):
    fails = llm_df[~llm_df['correct'] & (llm_df['llm_pred'] != -1)]
    sample = fails.sample(min(n, len(fails)), random_state=SEED)
    print(f'=== LLM FAILURES [{domain.upper()}] ({len(fails)} total) ===')
    for i, (_, row) in enumerate(sample.iterrows()):
        gt   = 'POS' if row['ground_truth'] == 1 else 'NEG'
        pred = 'POS' if row['llm_pred'] == 1 else 'NEG'
        print(f'[{i+1}] Truth:{gt} → LLM:{pred} | raw:"{row["llm_label_raw"]}" | words:{row["word_count"]}')
        print(f'  {str(row["text"])[:300]}...')
        print()

for domain in DOMAINS:
    if domain in llm_results:
        show_llm_failures(llm_results[domain], domain)
        print('-' * 70)

### LLM Failure Analysis — Observed Patterns

Across 6,000 samples, GPT-4o-mini produced **6 API errors** on Amazon (marked `llm_pred=-1`) and the following error counts per domain:
- IMDb: 132 failures (6.6%)
- Twitter: 181 failures (9.1%)
- Amazon: 76 failures (3.8%)

I identify the following empirical failure signatures:

#### 1. IMDb Failures — Boundary Ambiguity and Rhetorical Complexity
The most common IMDb failure I observed involved **reviews that hedge positively before endorsing** — the model classified several clearly positive reviews as negative because the text spent substantial space acknowledging flaws ("it's not surprising that people...can find flaws in it") before delivering a positive conclusion. This reflects a form of over-calibration to balanced rhetorical structure: the model weights ambiguity heavily when it encounters mixed signals, even if the overall register is clearly positive. The 200-word truncation also contributed: at least several IMDb failures cut off exactly where the review's tone shifted.

#### 2. Twitter Failures — Sarcasm and Event-Dependent Polarity
Twitter failures were most concentrated on **implicit polarity and ironic expression**. Tweets expressing frustration at missing a positive event ("I don't want to go to milan now") were frequently misclassified as positive because the surface language pattern-matches to travel enthusiasm. The model's zero-shot reasoning produces a plausible inference — but without the conversational thread, the ground-truth negative is unreachable. I note that even with Twitter as the second-hardest domain for the LLM (91.0%), the LLM still dramatically outperforms BERT (79.1%) here — a 11.8 pp gap that represents the largest method advantage in this study.

#### 3. Amazon Failures — Label Boundary Cases
Amazon failures were dominated by **mixed-quality reviews at the boundary of the positive/negative label split**. A representative example: `"Good, but not 'Out of the Silent Planet'. Not nearly as readable...The ending is a little heavy handed...It's still a great book"` — ground-truth positive (high star rating), LLM predicts negative (raw output: `mixed`). The model's reading is semantically correct — the review is qualified praise — but the label convention maps qualified positive reviews to the positive class. Amazon was simultaneously the LLM's **strongest domain** (96.2%) and the domain with the most interpretable failures.

---

**Comparison with BERT:** LLM failures feel semantically coherent — the model is wrong in ways that make interpretive sense given the text alone. BERT failures tend to reflect keyword anchoring on surface features without contextual reasoning. This qualitative difference is observable even when aggregate accuracy numbers are relatively close (as on IMDb: 93.4% vs. 89.2%).

## Disagreement Analysis — BERT vs LLM

I load the BERT results and compute per-domain disagreements between the two methods.

In [ ]:
disagreements = {}

for domain in DOMAINS:
    bert_path = f'../results/bert_{domain}.csv'
    if not Path(bert_path).exists():
        print(f'BERT results not found for {domain} — run bert_baseline.ipynb first')
        continue

    bert_df = pd.read_csv(bert_path)
    llm_df  = llm_results[domain]

    disagree_mask = (bert_df['bert_pred'].values != llm_df['llm_pred'].values)
    disagree_mask &= (llm_df['llm_pred'].values != -1)

    n_disagree = disagree_mask.sum()
    print(f'{domain.upper()}: {n_disagree}/{len(bert_df)} disagreements ({n_disagree/len(bert_df):.1%})')
    disagreements[domain] = disagree_mask

### Conclusions

Based on running GPT-4o-mini zero-shot across 6,000 samples (2,000 per domain), I draw the following conclusions:

1. **GPT-4o-mini outperforms DistilBERT on every domain without any domain-specific adaptation.** The accuracy advantages are 4.2 pp on IMDb, 11.8 pp on Twitter, and 7.4 pp on Amazon. The largest gain — Twitter — is also the domain where BERT struggles most, suggesting the LLM's broader contextual reasoning directly compensates for BERT's register brittleness.

2. **The LLM generalizes more robustly across domains.** The accuracy spread across domains is 5.2 pp for GPT-4o-mini (91.0%–96.2%) versus 10.1 pp for DistilBERT (79.1%–89.2%). This tighter range indicates that the LLM's performance is less sensitive to domain register shift — a meaningful practical advantage in systems that must handle diverse text sources.

3. **Amazon is GPT-4o-mini's strongest domain at 96.2%.** This is counterintuitive given Amazon's mixed-aspect structure, but I attribute it to the model's ability to follow the dominant evaluative thread in functional product reviews even when individual aspects carry different polarities.

4. **The total cost of $0.1334 for 6,000 samples is negligible at research scale.** At $0.022 per 1,000 samples average, LLM inference is financially competitive with any cloud-hosted NLP service. The constraint is latency (795–1,109 ms/sample), not cost — a deployment consideration that BERT's sub-150ms latency does not share.

5. **6 Amazon API errors (0.3%) occurred during inference.** All were handled gracefully and excluded from accuracy calculations. In production, a retry mechanism with exponential backoff would eliminate these.

---

*Results saved to `results/llm_{domain}.csv` for use in `cross_domain_analysis.ipynb`.*